In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import mediapipe as mp
import cv2

In [2]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

def media_detect(image , model):
    image = cv2.cvtColor(image , cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image , cv2.COLOR_RGB2BGR)
    return image , results


In [3]:
def draw_landmarks(image , results):
    mp_drawing.draw_landmarks(image , results.face_landmarks , mp_holistic.FACEMESH_CONTOURS,
                              mp_drawing.DrawingSpec(color=(80 ,110 ,10) , thickness=1 , circle_radius=1),
                                mp_drawing.DrawingSpec(color=(80 ,256 ,121) , thickness=1 , circle_radius=1)
                                )
    mp_drawing.draw_landmarks(image , results.pose_landmarks , mp_holistic.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(80 ,22 ,10) , thickness=2 , circle_radius=4),
                                mp_drawing.DrawingSpec(color=(80 ,44 ,121) , thickness=2 , circle_radius=2)
                                )
    mp_drawing.draw_landmarks(image , results.left_hand_landmarks , mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(121 ,22 ,76) , thickness=2 , circle_radius=4),
                                mp_drawing.DrawingSpec(color=(121 ,44 ,250) , thickness=2 , circle_radius=2)
                                )
    mp_drawing.draw_landmarks(image , results.right_hand_landmarks , mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(245 ,117 ,66) , thickness=2 , circle_radius=4),
                                mp_drawing.DrawingSpec(color=(245 ,66 ,230) , thickness=2 , circle_radius=2)
                                )

In [4]:
# mp_drawing.draw_landmarks??

In [8]:
cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5 , min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret , frame = cap.read()
        image , results = media_detect(frame , holistic)
        print(results)
        draw_landmarks(image , results)
        image = cv2.resize(image, (640, 360))

        cv2.imshow('Raw Webcam Feed' , image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

d:\Miniconda\envs\testenv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

In [9]:
results.pose_landmarks.landmark[0]

x: 0.499315888
y: 0.493579179
z: -0.751279712
visibility: 0.999954045

In [10]:
def extract_keypoints(results):
    pose = np.array([[res.x , res.y , res.z , res.visibility] for res in results.pose_landmarks.landmark]).flatten()  if results.left_hand_landmarks else np.zeros(132)
    face = np.array([[res.x , res.y , res.z] for res in results.face_landmarks.landmark]).flatten()  if results.face_landmarks else np.zeros(468 * 3)
    lh = np.array([[res.x , res.y , res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) 
    rh = np.array([[res.x , res.y , res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose , face , lh , rh])

In [11]:
extract_keypoints(results)

array([0., 0., 0., ..., 0., 0., 0.])

In [12]:
extract_keypoints(results).shape

(1662,)

In [13]:
extract_keypoints(results)[:-10]

array([0., 0., 0., ..., 0., 0., 0.])

In [6]:
data_path = os.path.join("mp_data")
actions = np.array(["hello" , "thanks" , "iloveyou"])
no_sequences = 30
sequence_length = 30

In [7]:
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(data_path , action , str(sequence)))
        except:
            pass

In [14]:
cap = cv2.VideoCapture(0)

cv2.namedWindow("OpenCv", cv2.WINDOW_NORMAL)
cv2.resizeWindow("OpenCv", 400, 300)

with mp_holistic.Holistic(min_detection_confidence=0.5 , min_tracking_confidence=0.5) as holistic:
    for action in actions:
        for sequence in range(no_sequences):
            for frame_num in range(sequence_length):
                ret , frame = cap.read()
                image , results = media_detect(frame , holistic)
                draw_landmarks(image , results)

                if frame_num == 0:
                    cv2.putText(image , "Starting Colliction" , (120,200),
                                cv2.FONT_HERSHEY_COMPLEX , 1 , (0,255,0) , 4 , cv2.LINE_AA)
                    cv2.putText(image , f"Colliction Frames for {action} video Number {sequence}" , (15,12),
                                cv2.FONT_HERSHEY_COMPLEX , 0.5 , (0,0,255) , 1 , cv2.LINE_AA)
                    cv2.imshow("OpenCv" , image)
                    cv2.waitKey(2000)

                else:
                    cv2.putText(image , f"Colliction Frames for {action} video Number {sequence}" , (15,12),
                                cv2.FONT_HERSHEY_COMPLEX , 0.5 , (0,0,255) , 1 , cv2.LINE_AA)
                    cv2.imshow("OpenCv" , image)


                keypoints = extract_keypoints(results)
                npy_path = os.path.join(data_path , action , str(sequence) , str(frame_num))
                np.save(npy_path , keypoints)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break


cap.release()
cv2.destroyAllWindows()

In [15]:
label_map = {label:num for num , label in enumerate(actions)}

In [16]:
label_map

{np.str_('hello'): 0, np.str_('thanks'): 1, np.str_('iloveyou'): 2}

In [17]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [18]:
sequences , labels = [] , []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(data_path , action , str(sequence) , "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [19]:
np.array(sequences).shape

(90, 30, 1662)

In [20]:
np.array(labels).shape

(90,)

In [21]:
x = np.array(sequences)

In [22]:
x.shape

(90, 30, 1662)

In [23]:
x

array([[[ 0.        ,  0.        ,  0.        , ...,  0.439578  ,
          0.44163376,  0.01342394],
        [ 0.        ,  0.        ,  0.        , ...,  0.39918941,
          0.3860727 , -0.00182223],
        [ 0.        ,  0.        ,  0.        , ...,  0.39534375,
          0.38360074, -0.00681015],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  0.19460493,
          0.2031845 , -0.03555139],
        [ 0.        ,  0.        ,  0.        , ...,  0.18623492,
          0.19575474, -0.03107021],
        [ 0.        ,  0.        ,  0.        , ...,  0.11779102,
          0.20267694, -0.01857933]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.11582428,
          0.24235997, -0.03879413],
        [ 0.        ,  0.        ,  0.        , ...,  0.43826926,
          0.45759177,  0.01969549],
        [ 0.        ,  0.        ,  0.        , ...,  0.4361673 ,
          0.45320517,  0.00381807],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  

In [24]:
y = to_categorical(labels).astype(int)

In [25]:
y.shape

(90, 3)

In [26]:
xtrain , xtest , ytrain , ytest = train_test_split(x, y, test_size=0.2)

In [27]:
xtrain.shape , ytrain.shape

((72, 30, 1662), (72, 3))

In [28]:
xtest.shape , ytest.shape

((18, 30, 1662), (18, 3))

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , LSTM
from tensorflow.keras.callbacks import TensorBoard 

In [30]:
log_dir = os.path.join("Logs")
tb_callbacks = TensorBoard(log_dir=log_dir)

In [31]:
model = Sequential()
model.add(LSTM(64 , return_sequences=True , activation='relu',input_shape=(30,1662)))
model.add(LSTM(128 , return_sequences=True , activation='relu'))
model.add(LSTM(64 , return_sequences=False , activation='relu'))
model.add(Dense(64 , activation='relu'))
model.add(Dense(32 , activation='relu'))
model.add(Dense(actions.shape[0] , activation='softmax'))

d:\Miniconda\envs\testenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [32]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │       442,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 596,675 (2.28 MB)

 Trainable params: 596,675 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model.compile(optimizer='Adam' , loss='categorical_crossentropy' , metrics=['categorical_accuracy'])

In [34]:
model.fit(
    xtrain,
    ytrain,
    epochs=2000,
    callbacks=[tb_callbacks]
)

Epoch 1/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 15s 162ms/step - categorical_accuracy: 0.3333 - loss: 2.2755
Epoch 2/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - categorical_accuracy: 0.3333 - loss: 1.1251
Epoch 3/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - categorical_accuracy: 0.4167 - loss: 1.5099
Epoch 4/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - categorical_accuracy: 0.2222 - loss: 6.7904
Epoch 5/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - categorical_accuracy: 0.5000 - loss: 5.5854
Epoch 6/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - categorical_accuracy: 0.5000 - loss: 11.8306
Epoch 7/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - categorical_accuracy: 0.1667 - loss: 59.2464
Epoch 8/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - categorical_accuracy: 0.3889 - loss: 20.5338
Epoch 9/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - categorical_accuracy: 0.3889 - loss: 73.2224
Epoch 10/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - categorical_accuracy: 0.3611 - loss: 40.8392
Epoch 11/2000

In [35]:
model.predict(xtest)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 923ms/step


array([[9.7509062e-01, 7.2885897e-09, 2.4909379e-02],
       [1.9271480e-09, 9.9999976e-01, 2.5069369e-07],
       [9.9999011e-01, 1.5913425e-08, 9.8796336e-06],
       [1.4061190e-05, 9.9500424e-01, 4.9817511e-03],
       [1.4757322e-07, 3.7981368e-07, 9.9999952e-01],
       [4.8632178e-21, 1.2717890e-16, 1.0000000e+00],
       [1.0909602e-09, 2.3983293e-09, 1.0000000e+00],
       [9.9999988e-01, 1.7240367e-07, 1.2188060e-08],
       [4.0350812e-09, 8.5570404e-09, 1.0000000e+00],
       [9.9989212e-01, 6.8121159e-10, 1.0787392e-04],
       [9.9999678e-01, 1.4587223e-10, 3.2570931e-06],
       [9.9999881e-01, 1.1857376e-06, 5.5832685e-09],
       [9.5596517e-09, 9.9999893e-01, 1.0835738e-06],
       [3.6878831e-09, 9.9999964e-01, 3.6331465e-07],
       [3.0700020e-10, 1.0000000e+00, 1.9064130e-08],
       [1.8628215e-09, 9.9999976e-01, 1.8702890e-07],
       [4.4262404e-07, 9.9999952e-01, 7.4029849e-09],
       [9.9994934e-01, 1.3461428e-09, 5.0648869e-05]], dtype=float32)

In [36]:
model.save('action_new3.h5')

In [37]:
from sklearn.metrics import multilabel_confusion_matrix , accuracy_score

ypred_test = model.predict(xtest)
ypred_train = model.predict(xtrain)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step 


In [38]:
ytrue_test = np.argmax(ytest , axis=1).tolist()
ytrue_train = np.argmax(ytrain , axis=1).tolist()

In [39]:
ypred_test = np.argmax(ypred_test , axis=1).tolist()
ypred_train = np.argmax(ypred_train , axis=1).tolist()

In [40]:
multilabel_confusion_matrix(ytrue_test , ypred_test)

array([[[11,  1],
        [ 0,  6]],

       [[11,  1],
        [ 0,  6]],

       [[12,  0],
        [ 2,  4]]])

In [41]:
multilabel_confusion_matrix(ytrue_train , ypred_train)

array([[[48,  0],
        [ 0, 24]],

       [[48,  0],
        [ 0, 24]],

       [[48,  0],
        [ 0, 24]]])

In [42]:
print(accuracy_score(ytrue_test , ypred_test))
print(accuracy_score(ytrue_train , ypred_train))

0.8888888888888888
1.0


In [43]:
res = model.predict(xtest)
actions[np.argmax(res[4])]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step


np.str_('iloveyou')

In [44]:
actions[np.argmax(ytest[4])]

np.str_('iloveyou')

In [45]:
ytest

array([[0, 0, 1],
       [0, 1, 0],
       [1, 0, 0],
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       [1, 0, 0],
       [0, 0, 1],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0],
       [1, 0, 0]])

In [46]:
res

array([[9.7509062e-01, 7.2885897e-09, 2.4909379e-02],
       [1.9271480e-09, 9.9999976e-01, 2.5069369e-07],
       [9.9999011e-01, 1.5913425e-08, 9.8796336e-06],
       [1.4061190e-05, 9.9500424e-01, 4.9817511e-03],
       [1.4757322e-07, 3.7981368e-07, 9.9999952e-01],
       [4.8632178e-21, 1.2717890e-16, 1.0000000e+00],
       [1.0909602e-09, 2.3983293e-09, 1.0000000e+00],
       [9.9999988e-01, 1.7240367e-07, 1.2188060e-08],
       [4.0350812e-09, 8.5570404e-09, 1.0000000e+00],
       [9.9989212e-01, 6.8121159e-10, 1.0787392e-04],
       [9.9999678e-01, 1.4587223e-10, 3.2570931e-06],
       [9.9999881e-01, 1.1857376e-06, 5.5832685e-09],
       [9.5596517e-09, 9.9999893e-01, 1.0835738e-06],
       [3.6878831e-09, 9.9999964e-01, 3.6331465e-07],
       [3.0700020e-10, 1.0000000e+00, 1.9064130e-08],
       [1.8628215e-09, 9.9999976e-01, 1.8702890e-07],
       [4.4262404e-07, 9.9999952e-01, 7.4029849e-09],
       [9.9994934e-01, 1.3461428e-09, 5.0648869e-05]], dtype=float32)

In [47]:
np.argmax(res)

np.int64(17)

In [48]:
np.argmax(res[0])

np.int64(0)

In [49]:
actions[np.argmax(res[0])]

np.str_('hello')

In [51]:
res[np.argmax(res)]

np.float32(0.99974495)

In [63]:
sequence = []
sentance = []
threshold = 0.7

cap = cv2.VideoCapture(0)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('my_testing_video.mp4', fourcc, 20.0, (640, 360))

with mp_holistic.Holistic(min_detection_confidence=0.5 , min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret , frame = cap.read()
        image , results = media_detect(frame , holistic)
        draw_landmarks(image , results)

        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        if len(sequence) > 30:
            sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence , axis=0))[0]
            idx = np.argmax(res)
            word = actions[idx]
            pre = res[idx]

            print(word)

            if pre > threshold:
                if len(sentance) > 0:
                    if word != sentance[-1]:
                        sentance.append(actions[np.argmax(res)])
                else:
                    sentance.append(actions[np.argmax(res)])

            if len(sentance) > 5:
                sentance = sentance[-5:]


            if results.pose_landmarks:
                h , w , c = image.shape
                nose = results.pose_landmarks.landmark[0]
                cx , cy = int(nose.x*w) , int(nose.y*h)
                rech , recw = 40 , 150
                
                x1 = int(cx - (recw // 2))
                y1 = int(cy - 140)
                x2 = x1 + recw
                y2 = y1 + rech

                txt = f"{word} {pre:.4f}"
                color = (245, 117, 16) if pre > threshold else (0, 0, 255)

            cv2.rectangle(image , (x1 , y1) , (x2 , y2) , color , -1)
            cv2.putText(image , txt , (x1+10 , y1+30) , cv2.FONT_HERSHEY_SIMPLEX , 0.8 , (255, 255, 255),
                        2 , cv2.LINE_AA)

        image = cv2.resize(image, (640, 360))
        out.write(image)
        cv2.imshow('OpenCv' , image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
iloveyou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
iloveyou
1/1 ━━━━━━━━